In [1]:
import requests 
from flask import Flask, request, jsonify, render_template
import pandas as pd
import pyodbc

In [ ]:
df = pd.read_csv("customers.csv")
cst = pd.DataFrame(df)

In [3]:
cst = cst.drop_duplicates(['first_name'])
cst

,customer_id,first_name,last_name,city,age,email
0,1,Ahmed,Hassan,Cairo,25,ahmed@email.com
1,2,Sara,Ali,Giza,31,sara@email.com
2,3,Omar,Mohamed,cairo,17,omar@email.com
3,4,Mona,Ibrahim,Alexandria,28,NaN


In [4]:
cst['email'] = cst['email'].fillna('no email provided')
cst

,customer_id,first_name,last_name,city,age,email
0,1,Ahmed,Hassan,Cairo,25,ahmed@email.com
1,2,Sara,Ali,Giza,31,sara@email.com
2,3,Omar,Mohamed,cairo,17,omar@email.com
3,4,Mona,Ibrahim,Alexandria,28,no email provided


In [5]:
cst['Full_name'] = cst['first_name'] + ' ' + cst['last_name']

cst['city'] = cst['city'].str.strip().str.title()

cst['age'] = pd.to_numeric(cst['age'], errors='coerce')

cst

,customer_id,first_name,last_name,city,age,email,Full_name
0,1,Ahmed,Hassan,Cairo,25,ahmed@email.com,Ahmed Hassan
1,2,Sara,Ali,Giza,31,sara@email.com,Sara Ali
2,3,Omar,Mohamed,Cairo,17,omar@email.com,Omar Mohamed
3,4,Mona,Ibrahim,Alexandria,28,no email provided,Mona Ibrahim


In [6]:
cst.drop(['first_name', 'last_name'],axis = 1 , inplace=True)
cst

,customer_id,city,age,email,Full_name
0,1,Cairo,25,ahmed@email.com,Ahmed Hassan
1,2,Giza,31,sara@email.com,Sara Ali
2,3,Cairo,17,omar@email.com,Omar Mohamed
3,4,Alexandria,28,no email provided,Mona Ibrahim


In [7]:
df1 = pd.read_parquet("orders.parquet")
df1= pd.DataFrame(df1)
orders = df1.copy()

In [8]:
orders= orders.drop_duplicates(['order_id'])

orders['order_date'] = pd.to_datetime(orders['order_date'], errors='coerce')

orders['quantity'] = pd.to_numeric(orders['quantity'], errors='coerce')

orders['customer_id'] = pd.to_numeric(orders['customer_id'], errors='coerce')

orders['product_id'] = pd.to_numeric(orders['product_id'], errors='coerce')

orders

,order_id,customer_id,product_id,quantity,order_date
0,1001,1,101,2,2026-09-01
1,1002,2,102,1,2026-09-01
2,1003,3,103,3,2026-09-02
3,1004,4,101,1,2026-09-02
4,1005,1,104,2,2026-09-03
5,1006,5,102,1,2026-09-03
6,1007,2,105,4,2026-09-04
7,1008,3,101,1,2026-09-04
8,1009,4,103,2,2026-09-05
9,1010,1,104,1,2026-09-05


In [9]:
url = "https://raw.githubusercontent.com/MohammedHameds/test1455/refs/heads/main/products.csv"

products = pd.read_csv(url)
products = pd.DataFrame(products)
products

,product_id,product_name,category,unit_price
0,101,Laptop,Electronics,25000
1,102,Mouse,Accessories,500
2,103,Keyboard,Accessories,1200
3,104,Monitor,Electronics,7000
4,105,Headphones,Accessories,2500
5,106,Webcam,Accessories,1800


In [10]:
products = products.drop_duplicates()
products

,product_id,product_name,category,unit_price
0,101,Laptop,Electronics,25000
1,102,Mouse,Accessories,500
2,103,Keyboard,Accessories,1200
3,104,Monitor,Electronics,7000
4,105,Headphones,Accessories,2500
5,106,Webcam,Accessories,1800


In [11]:
products.columns

Index(['product_id', 'product_name', 'category', 'unit_price'], dtype='str')

In [12]:
products.columns = products.columns.str.strip()

products['product_name']  = products['product_name'].str.strip()
products['category']  = products['category'].str.strip()


In [13]:
products ['unit_price'] =  pd.to_numeric(products['unit_price'], errors='coerce')

In [14]:
sales = orders.merge(cst, on="customer_id", how="inner")

sales = sales.merge(products, on="product_id", how="inner")

sales["total_price"] = sales["quantity"] * sales["unit_price"]

sales

,order_id,customer_id,product_id,quantity,order_date,city,age,email,Full_name,product_name,category,unit_price,total_price
0,1001,1,101,2,2026-09-01,Cairo,25,ahmed@email.com,Ahmed Hassan,Laptop,Electronics,25000,50000
1,1002,2,102,1,2026-09-01,Giza,31,sara@email.com,Sara Ali,Mouse,Accessories,500,500
2,1003,3,103,3,2026-09-02,Cairo,17,omar@email.com,Omar Mohamed,Keyboard,Accessories,1200,3600
3,1004,4,101,1,2026-09-02,Alexandria,28,no email provided,Mona Ibrahim,Laptop,Electronics,25000,25000
4,1005,1,104,2,2026-09-03,Cairo,25,ahmed@email.com,Ahmed Hassan,Monitor,Electronics,7000,14000
5,1007,2,105,4,2026-09-04,Giza,31,sara@email.com,Sara Ali,Headphones,Accessories,2500,10000
6,1008,3,101,1,2026-09-04,Cairo,17,omar@email.com,Omar Mohamed,Laptop,Electronics,25000,25000
7,1009,4,103,2,2026-09-05,Alexandria,28,no email provided,Mona Ibrahim,Keyboard,Accessories,1200,2400
8,1010,1,104,1,2026-09-05,Cairo,25,ahmed@email.com,Ahmed Hassan,Monitor,Electronics,7000,7000


Final columns:

```text
order_id
customer
product
quantity
unit_price
total_amount
```

In [15]:
sales1 = sales[
    [
        "order_id",
        "Full_name",
        "product_name",
        "quantity",
        "unit_price",
        "total_price",
    ]]

sales1

,order_id,Full_name,product_name,quantity,unit_price,total_price
0,1001,Ahmed Hassan,Laptop,2,25000,50000
1,1002,Sara Ali,Mouse,1,500,500
2,1003,Omar Mohamed,Keyboard,3,1200,3600
3,1004,Mona Ibrahim,Laptop,1,25000,25000
4,1005,Ahmed Hassan,Monitor,2,7000,14000
5,1007,Sara Ali,Headphones,4,2500,10000
6,1008,Omar Mohamed,Laptop,1,25000,25000
7,1009,Mona Ibrahim,Keyboard,2,1200,2400
8,1010,Ahmed Hassan,Monitor,1,7000,7000


In [17]:
sales1 = sales1.rename(
    columns={
        "Full_name": "customer",
        "product_name": "product"
    }
)

sales1

,order_id,customer,product,quantity,unit_price,total_price
0,1001,Ahmed Hassan,Laptop,2,25000,50000
1,1002,Sara Ali,Mouse,1,500,500
2,1003,Omar Mohamed,Keyboard,3,1200,3600
3,1004,Mona Ibrahim,Laptop,1,25000,25000
4,1005,Ahmed Hassan,Monitor,2,7000,14000
5,1007,Sara Ali,Headphones,4,2500,10000
6,1008,Omar Mohamed,Laptop,1,25000,25000
7,1009,Mona Ibrahim,Keyboard,2,1200,2400
8,1010,Ahmed Hassan,Monitor,1,7000,7000


In [18]:
pyodbc.drivers()

['SQL Server',
 'ODBC Driver 17 for SQL Server',
 'ODBC Driver 18 for SQL Server']

In [24]:
conn = pyodbc.connect(
    "driver={ODBC Driver 17 for SQL Server};"
    "Server=localhost;"
    "Database=salesDB;"
    "Trusted_Connection=yes;"    
)

cursor = conn.cursor()

cursor.execute(
    """
    create table sales1 (
        order_id INT,
        customer varchar(255),
        product varchar(255),
        quantity int,
        unit_price float,
        total_price float
    )
"""
)

cursor = cursor.executemany(
    "insert into sales1 (order_id, customer, product, quantity, unit_price, total_price) values (?, ?, ?, ?, ?, ?)",
    sales1.values.tolist()
)

conn.commit()
conn.close()